# Minggu 4 — Praktik: Memodelkan Head Wave dan Menguji Hukum Snell

**Seismologi PAGF262413** · Program Studi Sarjana Geofisika FMIPA UGM

Kalian memegang seismogram sintetik **24 geofon** dari satu tembakan di permukaan, di atas
model dua lapis mendatar. Rekaman itu memuat tiga gelombang sekaligus: **langsung**, **pantul**,
dan **kepala** (*head wave*).

Tugasnya satu kalimat: **pulihkan model bumi dari rekaman, lalu uji apakah Hukum Snell benar-benar
meramalkan bentuk gelombang yang disimulasikan.**

Kalian **tidak diberi** nilai $V_1$, $V_2$, dan $h$. Angka itu ada di `data/W04_model_sejati.json`
— jangan dibuka sebelum Bagian 4. Membukanya lebih dulu menghilangkan seluruh gunanya.

| Isian wajib | |
|:--|:--|
| Nama | |
| NIM | |
| Kelompok & peran | |

> **Aturan angka mata kuliah ini:** setiap hasil dilaporkan sebagai **selang keyakinan 80%**,
> bukan titik tunggal. Tulis juga dari mana lebar selang itu kalian peroleh.

## Bagian 0 — Prediksi terkunci

**Kerjakan sebelum menjalankan sel apa pun.** Ini instrumen 4 pada `Asesmen_Era_AI_Seismologi.md`;
yang dinilai penalarannya, bukan ketepatan angkanya.

1. Pada rekaman 24 geofon, gelombang mana yang tiba lebih dulu di **offset dekat**? Mengapa?
2. Adakah offset yang membuat urutannya berbalik? Apa nama jarak itu?
3. Gambar sketsa kasar kurva waktu tempuh yang kalian harapkan — dua garis atau satu?
4. Gelombang kepala merambat di batas dengan kecepatan lapisan bawah, tetapi amplitudonya
   meluruh lebih cepat daripada gelombang langsung. Tebak alasannya.

✍️ *(tulis di sini, lalu jangan diubah lagi)*

## Bagian 1 — Memuat rekaman

Sumber data: `data/W04_seismogram_dua_lapis.npz`, dihasilkan `skrip/w04_gelombang_dua_lapis.py`.
Kalau kalian sudah menjalankan SPECFEM2D sendiri (`skrip/w04_specfem2d/`), pakai keluaran kalian:
sel kedua di bawah membaca berkas `.semv` ASCII.

In [ ]:
import numpy as np, matplotlib.pyplot as plt, json
plt.rcParams['figure.figsize'] = (7, 6); plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = .25

d    = np.load('data/W04_seismogram_dua_lapis.npz')
seis = d['seis']          # (nt, 24) kecepatan partikel vertikal
dt   = float(d['dt'])     # langkah waktu (s)
t0   = float(d['t0'])     # pergeseran puncak wavelet Ricker (s)
offs = d['offs']          # offset tiap geofon (m)
f0   = float(d['f0'])

t = np.arange(seis.shape[0])*dt - t0     # t = 0 pada waktu asal sumber
print(f'{seis.shape[1]} geofon, offset {offs.min():.0f}-{offs.max():.0f} m, '
      f'spasi {offs[1]-offs[0]:.0f} m')
print(f'{seis.shape[0]} cuplikan, dt = {dt*1e3:.3f} ms, rekaman {t[-1]*1e3:.0f} ms, f0 = {f0:.0f} Hz')

*Opsional — kalau kalian menjalankan SPECFEM2D sendiri:*

In [ ]:
# def muat_specfem(folder='OUTPUT_FILES', n=24):
#     tr = [np.loadtxt(f'{folder}/AA.S{i+1:04d}.BXZ.semv') for i in range(n)]
#     t_ = tr[0][:, 0]
#     return np.column_stack([x[:, 1] for x in tr]), t_[1]-t_[0], t_[0]
# seis, dt, tmin = muat_specfem()

## Bagian 2 — Menggambar penampang rekaman

Plot 24 jejak berdampingan terhadap offset (*wiggle plot*). Beri penguatan terhadap waktu,
kalau tidak tiba-awal di offset jauh akan tenggelam oleh energi belakangan yang jauh lebih kuat.

In [ ]:
gain = 1 + 12*np.maximum(t, 0)         # penguat sederhana; coba ubah angkanya
sk   = 0.6*(offs[1]-offs[0])

plt.figure(figsize=(8, 7))
for i, off in enumerate(offs):
    tr = seis[:, i]*gain; tr = tr/np.abs(tr).max()
    plt.plot(off + sk*tr, t*1e3, 'k-', lw=.6)
    plt.fill_betweenx(t*1e3, off, off + sk*tr, where=tr > 0, color='k', alpha=.55, lw=0)
plt.gca().invert_yaxis(); plt.ylim(185, -25)
plt.xlabel('offset (m)'); plt.ylabel('waktu (ms)')
plt.title('Penampang rekaman 24 geofon');

✍️ **Jawaban 1.** Lihat baik-baik sebelum menghitung apa pun.

1. Berapa banyak kedatangan berbeda yang bisa kalian bedakan? Tandai perkiraan offset dan waktunya.
2. Pada offset berapa kira-kira kemiringan tiba-pertama **berubah**? Itu tebakan awal $X_c$ kalian.
3. Adakah kedatangan yang bentuk kurvanya melengkung, bukan lurus? Menurut kalian gelombang apa itu?

*(tulis di sini)*

## Bagian 3 — Memetik *first break*

Pemetikan bukan inti pelajaran hari ini, jadi pemetiknya diberikan. Dua langkahnya:

1. **Ambang amplitudo** — sampel pertama yang melampaui 1% amplitudo maksimum jejak itu.
2. **Dinaikkan ke puncak wavelet** — ambang terlewati di kaki gelombang, bukan di puncaknya.

Ambangnya sengaja **sangat rendah**, dan itu bukan kecerobohan: **gelombang kepala amplitudonya
jauh lebih kecil** daripada gelombang langsung. Naikkan `frac` ke 0,05 lalu jalankan ulang — kalian
akan melihat sendiri pick di offset jauh melompat ke kedatangan yang salah.

Rekaman ini bebas derau sehingga ambang sesederhana ini memadai. Pada data lapangan yang berderau
ambang tetap akan gagal; di sana dipakai STA/LTA atau AIC.

In [ ]:
def petik(seis, dt, t0, f0, frac=0.01):
    """First break: ambang amplitudo, lalu dinaikkan ke puncak wavelet."""
    t    = np.arange(seis.shape[0])*dt - t0
    naik = int(0.7/f0/dt)                     # jendela naik: 0,7 perioda
    out  = []
    for i in range(seis.shape[1]):
        a = np.abs(seis[:, i].astype(float))
        j = int(np.argmax(a > frac*a.max()))  # lewati ambang pertama kali
        out.append(t[j + int(np.argmax(a[j:j+naik]))])
    return np.array(out)

pick = petik(seis, dt, t0, f0)
for o, p in zip(offs, pick): print(f'{o:6.0f} m   {p*1e3:8.2f} ms')

In [ ]:
plt.figure(figsize=(7, 5))
plt.plot(offs, pick*1e3, 'o', mfc='yellow', mec='r')
plt.xlabel('offset (m)'); plt.ylabel('waktu tiba (ms)')
plt.title('Kurva waktu tempuh tiba-pertama');

✍️ **Jawaban 2.** Titik-titik itu jelas **tidak** jatuh pada satu garis lurus.
Di offset berapa patahannya? Bandingkan dengan tebakan kalian di Jawaban 1.

## Bagian 4 — Inversi dua garis

Sekarang bagian intinya. Dari Minggu 4:

$$T_\text{langsung} = \frac{X}{V_1}, \qquad T_\text{kepala} = \frac{X}{V_2} + t_i, \qquad
  t_i = \frac{2h\sqrt{V_2^2-V_1^2}}{V_1 V_2}$$

### Dua hal yang harus kalian tangani lebih dulu

**Geofon di sekitar crossover harus dibuang.** Di sana gelombang langsung dan gelombang kepala tiba
nyaris bersamaan dan saling mengganggu, sehingga pick-nya tidak mewakili cabang mana pun. Buang
kira-kira dua geofon di kiri dan satu di kanan titik crossover.

**Periksa statiknya.** Garis gelombang langsung secara fisis **wajib** lewat titik asal: pada
offset nol, waktu tempuhnya nol. Kalau intersepnya tidak nol, selisih itu adalah statik — waktu
pemicu, kedalaman sumber, atau pergeseran fasa wavelet. Statik **tidak** mengubah kemiringan
(jadi $V_1$ dan $V_2$ aman) tetapi **mengubah** intersep cabang refraksi, sehingga kedalaman $h$
ikut salah bila statik dibiarkan.

In [ ]:
XC_TEBAKAN = ...      # <-- TUGAS: isi dari Jawaban 1/2, dalam meter

dekat = offs < XC_TEBAKAN - 15      # buang zona interferensi di sekitar crossover
jauh  = offs > XC_TEBAKAN + 10
print(f'{dekat.sum()} geofon cabang langsung, {jauh.sum()} geofon cabang refraksi, '
      f'{len(offs)-dekat.sum()-jauh.sum()} dibuang')

_, statik = np.polyfit(offs[dekat], pick[dekat], 1)
pk = pick - statik
print(f'statik = {statik*1e3:+.2f} ms  (dihapus dari semua pick)')

In [ ]:
s1, i1 = np.polyfit(offs[dekat], pk[dekat], 1)
s2, ti = np.polyfit(offs[jauh],  pk[jauh],  1)

V1 = ...      # <-- TUGAS: dari kemiringan s1
V2 = ...      # <-- TUGAS: dari kemiringan s2
h  = ...      # <-- TUGAS: balikkan rumus t_i untuk memperoleh h
ic = ...      # <-- TUGAS: sudut kritis, dalam derajat
Xc = ...      # <-- TUGAS: crossover distance dari V1, V2, h

print(f'V1 = {V1:7.1f} m/s\nV2 = {V2:7.1f} m/s\nh  = {h:7.1f} m'
      f'\nic = {ic:7.2f} derajat\nXc = {Xc:7.1f} m  (tebakan awal {XC_TEBAKAN:.0f} m)')

✍️ **Jawaban 3.**

1. Laporkan $V_1$, $V_2$, $h$, $i_c$ masing-masing **dengan selang keyakinan 80%**. Lebar selangnya
   dari mana? (Petunjuk: ubah `XC_TEBAKAN` sebesar ±1 geofon dan `frac` pemetik, lalu lihat berapa
   hasilnya bergeser. Itu ketidakpastian yang jujur, bukan tebakan.)
2. $X_c$ hasil hitungan cocok dengan tebakan mata dari grafik? Kalau meleset, dari mana selisihnya?
3. Berapa milidetik statiknya? Bandingkan dengan satu perioda wavelet $1/f_0$ = 10 ms. Menurut kalian
   apakah simulasi ini memang punya statik berarti, atau nilainya sekadar sisa pembulatan pemetik?

## Bagian 5 — Menguji Snell terhadap gelombang simulasi

Inilah pengujiannya. Kalian punya $V_1$, $V_2$, $h$ hasil inversi. Ramalkan **ketiga** kurva
waktu tempuh dari Hukum Snell, lalu tumpangkan ke seismogram yang sesungguhnya.

$$T_\text{pantul} = \frac{\sqrt{X^2 + 4h^2}}{V_1}$$

Gelombang kepala hanya ada di luar jarak kritis $X > 2h\tan i_c$ — di dalamnya tidak ada sinar
yang mencapai batas pada sudut kritis.

In [ ]:
X = np.linspace(0, offs.max()*1.05, 300)

T_langsung = ...      # <-- TUGAS
T_pantul   = ...      # <-- TUGAS
X_kritis   = ...      # <-- TUGAS: 2h tan(ic)
T_kepala   = np.where(X >= X_kritis, ..., np.nan)   # <-- TUGAS

plt.figure(figsize=(8, 7))
for i, off in enumerate(offs):
    tr = seis[:, i]*gain; tr = tr/np.abs(tr).max()
    plt.plot(off + sk*tr, (t + statik)*1e3, 'k-', lw=.6)
plt.plot(X, T_langsung*1e3, 'g-',  lw=2, label='langsung (Snell)')
plt.plot(X, T_kepala*1e3,   'r-',  lw=2, label='kepala (Snell)')
plt.plot(X, T_pantul*1e3,   'm--', lw=2, label='pantul (Snell)')
plt.axvline(Xc, color='b', ls=':', label=f'crossover {Xc:.0f} m')
plt.gca().invert_yaxis(); plt.ylim(185, -25)
plt.xlabel('offset (m)'); plt.ylabel('waktu (ms)')
plt.title('Ramalan Hukum Snell di atas gelombang simulasi'); plt.legend(loc='lower left');

✍️ **Jawaban 4 — pertanyaan terpenting di notebook ini.**

1. Apakah ketiga kurva Snell jatuh di atas kedatangan yang benar pada seismogram? Untuk
   masing-masing kurva, seberapa besar selisihnya dalam milidetik?
2. Kurva pantul **melengkung**, dua lainnya lurus. Jelaskan asal geometris kelengkungan itu.
3. Pada offset dekat, kurva pantul dan kurva langsung hampir berimpit di suatu titik. Di mana,
   dan mengapa mereka mendekat di situ?
4. Amati amplitudo gelombang kepala terhadap offset. Bandingkan peluruhannya dengan gelombang
   langsung. Apakah cocok dengan tebakan kalian di Bagian 0 nomor 4?
5. **Batas metodenya.** Andaikan ada lapisan ketiga yang lebih *lambat* di bawah batuan dasar.
   Adakah cara mendeteksinya dari rekaman ini? Pertahankan jawaban kalian.

## Bagian 6 — Uji terhadap model sejati

**Baru sekarang** kunci boleh dibuka.

In [ ]:
sejati = json.load(open('data/W04_model_sejati.json'))
for k, v in sejati.items(): print(f'{k:18s} {v:10.3f}')

for nama, saya, benar in [('V1', V1, sejati['V1']), ('V2', V2, sejati['V2']),
                          ('h',  h,  sejati['h']),
                          ('ic', ic, sejati['sudut_kritis_deg'])]:
    print(f'{nama:3s} saya {saya:9.2f}   sejati {benar:9.2f}   '
          f'galat {100*(saya-benar)/benar:+6.2f} %')

✍️ **Jawaban 5 — refleksi penutup.**

1. Besaran mana yang paling teliti kalian pulihkan, mana yang paling meleset? Mengapa justru itu?
2. Apakah nilai sejati masuk ke dalam selang keyakinan 80% kalian di Jawaban 3? Kalau **tidak**,
   itu berarti kalian terlalu percaya diri — dan itu jawaban yang jauh lebih menarik daripada
   kebetulan tepat. Bahas apa yang membuat kalian meremehkan ketidakpastiannya.
3. Simulasi ini memakai batas yang rata sempurna, medium tanpa derau, dan geofon yang tepat
   di tempatnya. Sebutkan **tiga** hal yang berbeda pada survei refraksi sungguhan, dan untuk
   masing-masing, ke arah mana ia akan membelokkan hasil kalian.
4. Satu kalimat: apa yang tadinya kalian kira paham tentang *head wave*, dan ternyata tidak?